# NanoScale-LM: train a language model from scratch

This notebook trains a decoder-only transformer end to end: **you build the tokenizer,
the model, and the optimizer from raw PyTorch**: nothing here calls a high-level trainer.

Two paths:

* **CPU path** (`nano`, ~5M params): finishes in about two minutes. No accelerator needed.
* **GPU path** (`micro`, ~40M params): streams FineWeb-Edu to the Chinchilla-optimal
  20:1 token budget. A few hours on a free T4. Set the runtime to GPU first
  (*Runtime → Change runtime type → T4 GPU*).

Everything runs on free hardware. There is no paid API, no gated dataset, and no
checkpoint download in this notebook; the weights you use at the end are the ones you
trained here.

## 0. Setup

In [ ]:
# Clone and install. ~90 seconds on a fresh Colab runtime.
!git clone --depth 1 https://github.com/vedant1711/nanoscale-lm 2>/dev/null || true
%cd nanoscale-lm
!pip install -q -e ".[data]"

import torch

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print(f"torch {torch.__version__} | gpu: {gpu}")

In [ ]:
from nanoscale.utils import hardware_string

print(hardware_string())

## 1. Train the tokenizer

Byte-level BPE, implemented in `src/nanoscale/tokenizer/bpe.py`. Two properties matter
and both are asserted by the test suite:

1. **Round-trip on arbitrary bytes.** Working over bytes rather than Unicode code points
   means `decode(encode(s)) == s` for *any* string, including emoji, mojibake and lone
   surrogates. There is no `<unk>` token because there cannot be one.
2. **Merges are learned by frequency, within pre-token boundaries.** The GPT-4
   pre-tokenization regex keeps merges from crossing word boundaries, which is what stops
   the vocabulary filling up with common phrases.

In [ ]:
from nanoscale.config import TokenizerConfig
from nanoscale.data.toy import generate_corpus
from nanoscale.tokenizer import BPETokenizer

corpus = generate_corpus(seed=1337, n_stories=20_000)
print(f"corpus: {len(corpus):,} chars")

tok = BPETokenizer.train(corpus, TokenizerConfig(vocab_size=1024))
print(tok)
print(f"compression: {tok.compression_ratio(corpus[:100_000]):.2f} bytes/token")

In [ ]:
# Round-trip anything.
for s in ["Lily went to the park.", "héllo wörld 🌍", "\x00\xff raw bytes", ""]:
    assert tok.decode(tok.encode(s)) == s, s
print("round-trip holds on all probes")

# Look at what it learned.
ids = tok.encode("Lily wanted to find a torn map.")
print([tok.decode([i]) for i in ids])

## 2. Build the model

Modern decoder-only stack, every piece written out in `src/nanoscale/model/`:

| Component | Why |
|---|---|
| **RMSNorm** | No mean subtraction, no bias: one reduction instead of two. |
| **Pre-norm residuals** | An identity path from embedding to logits, so depth is trainable. |
| **RoPE** | Relative position from a rotation, so no learned position table. |
| **GQA** | Fewer KV heads than Q heads, cuts the KV cache, which is what actually bounds serving. |
| **QK-norm** | Normalizes queries and keys *before* RoPE; keeps attention logits from exploding. |
| **SwiGLU** | Gated FFN; the gate lets the layer suppress its own activations. |
| **Zero-init output projections** | Every block starts as exactly the identity, so step 0 loss is exactly `ln(vocab)`. |

In [ ]:
from nanoscale.config import load_experiment
from nanoscale.model import build_model

cfg = load_experiment(tier="nano")
model = build_model(cfg.model)

for k, v in cfg.model.param_breakdown().items():
    print(f"{k:>16}: {v:>12,}")

In [ ]:
import math

import torch

# The zero-init claim, checked rather than asserted in prose:
# an untrained model should sit at exactly ln(vocab_size).
x = torch.randint(0, cfg.model.vocab_size, (2, 64))
with torch.no_grad():
    loss = model(x, targets=x).loss
print(f"untrained loss {loss.item():.4f}  vs  ln(vocab) = {math.log(cfg.model.vocab_size):.4f}")

## 3. The optimizer

Two optimizers, both written from scratch in `src/nanoscale/optim/`:

* **AdamW** for 1-D parameters (norms, biases, embeddings and the LM head).
* **Muon** for the 2-D hidden matrices. Muon takes the momentum buffer and
  **orthogonalizes** it with a Newton–Schulz iteration before applying it, so the update
  has a balanced spectrum instead of being dominated by its top singular direction.

The router that decides which parameter goes where is the interesting part: embeddings
and the head are *not* hidden matrices even though they are 2-D, and giving them to Muon
makes things worse.

In [ ]:
from nanoscale.optim import newton_schulz_orthogonalize, split_parameters

split = split_parameters(model)
print(f"Muon  : {sum(p.numel() for p in split.muon):>10,} parameters  {split.muon_names[:3]} ...")
print(f"AdamW : {sum(p.numel() for p in split.adamw):>10,} parameters  {split.adamw_names}")

# What orthogonalization does to a badly conditioned "gradient".
g = torch.randn(64, 64) @ torch.diag(torch.logspace(0, -4, 64)) @ torch.randn(64, 64)
o = newton_schulz_orthogonalize(g.clone())

for label, m in [("before", g), ("after ", o)]:
    s = torch.linalg.svdvals(m)
    print(f"{label}: sigma_max {s[0]:8.3f}  sigma_min {s[-1]:.2e}  cond {s[0] / s[-1]:12.1f}")

# Note what this does *not* say. Five NS steps do not produce a perfectly orthogonal
# matrix -- the smallest singular directions are still under-amplified. Muon does not
# need them to be: the point is to stop the update being dominated by its top singular
# direction, and a ~200x spectral compression already achieves that at a fraction of
# the cost of an SVD.

## 4. Pretrain, CPU path (`nano`)

400 steps. About 95 seconds on a laptop CPU, a little slower on a Colab CPU runtime.

In [ ]:
from nanoscale.train import Trainer

cfg = load_experiment(tier="nano", overrides=["train.device=cpu"])
trainer = Trainer(cfg, tokenizer=tok, out_dir="runs/nb/pretrain")
result = trainer.train()

print(
    f"\nfinal val loss {result.final_val_loss:.4f}  "
    f"(perplexity {math.exp(result.final_val_loss):.4f})"
)
print(
    f"{result.tokens:,} tokens in {result.wall_clock_s:.0f}s "
    f"at {result.tokens_per_second:,.0f} tokens/s"
)

In [ ]:
import matplotlib.pyplot as plt

steps = [r["step"] for r in result.history if "loss" in r]
loss = [r["loss"] for r in result.history if "loss" in r]
vsteps = [r["step"] for r in result.history if "val_loss" in r]
vloss = [r["val_loss"] for r in result.history if "val_loss" in r]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(steps, loss, lw=1, alpha=0.8, label="train")
ax.plot(vsteps, vloss, "o-", lw=1.5, label="val")
ax.axhline(math.log(cfg.model.vocab_size), ls=":", c="gray", label="ln(vocab) = chance")
ax.set(xlabel="step", ylabel="cross-entropy (nats/token)", title="nano tier")
ax.legend()
plt.show()

## 5. Generate

In [ ]:
from nanoscale.config import GenerateConfig
from nanoscale.serve import generate_text

out = generate_text(
    trainer.model,
    tok,
    "It was a sunny day. Lily went to the park with",
    GenerateConfig(max_new_tokens=96, temperature=0.8, top_p=0.95, seed=7),
)
print(out.text)

## 6. Pretrain, GPU path (`micro`)

Same code, larger config, real data. `micro` streams FineWeb-Edu (`sample-10BT`) so
nothing is downloaded up front, and trains to the full 20:1 token budget: ~808M tokens
for 40M parameters.

Expect a few hours on a free T4. Checkpoints are resumable: if Colab disconnects, rerun
the cell with `--resume` and it picks up at the exact batch it stopped on (that
"exact batch" is not decoration; resuming at the wrong offset silently costs you
validation loss, which is why `TokenBatcher.stream()` takes a starting offset).

In [ ]:
# Requires a GPU runtime. Skip on CPU.
if torch.cuda.is_available():
    !python -m nanoscale.cli train pretrain --tier micro -o runs/nb/micro
else:
    print("No GPU, switch the runtime to T4 to run the micro tier.")

## Where to go next

* `notebooks/colab_compress_and_serve.ipynb`: take this checkpoint and make it small and
  fast: distillation, GPTQ, speculative decoding.
* `nanoscale align sft <ckpt>` then `nanoscale align preference <ckpt> --method dpo`:
turn the base model into an instruction-following one.
* `docs/methodology.md`: every algorithm above with its formula, its citation, and the
  test that pins it.